# Free Steak Dinner VOR Model — 2026

Rebuilt from the 2025 notebook with these changes:

1. **Scoring uses actual Yahoo league settings** — half-PPR (0.5/rec), 6-pt passing TDs, −1 INT, rushing first downs
   now actually scored at 0.5 (was computed but commented out).
2. **Replacement levels computed programmatically every run** from roster
   structure (12 teams, 1QB/2RB/3WR/1TE + flex) — replaces the hardcoded
   `replacement_values = {...}` that went stale whenever FantasyPros updated.
3. **Consistent baseline** — replacement level is taken from the same
   points column that player VOR is computed from (the 2025 model
   subtracted unadjusted replacement values from adjusted player points).
4. **Header-name-based scraping** — columns are matched by their header text
   (including the PASSING/RUSHING/RECEIVING group row), so if FantasyPros
   reorders columns you get a loud error instead of silently wrong stats.
5. **Merge safety** — names are normalized (suffixes, punctuation, team
   aliases) before joining projections to ADP, and unmatched players are
   PRINTED instead of silently dropped.
6. **Half-PPR ADP page** (was scraping the full-PPR page) and a
   **VALUE = ADP rank − VOR rank** column: positive = the room will likely
   let this player fall past his real value in THIS league's scoring.
7. **K + DST included** (your league starts both) with approximated scoring.
8. **Gap-based tiers** per position, and **dated CSV snapshots** of every
   scrape so you can reproduce any board later (and backtest).



In [1]:
# ============================================================
# CELL 1 — LEAGUE CONFIG  (complete, self-contained)
# Free Steak Dinner NFL — Yahoo league 410984
# 12 teams · half-PPR · 6-pt pass TD · 0.5 rushing 1D
# ============================================================
import json
import re
from datetime import date

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Used by the FantasyPros ECR overlay in Cell 3.
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

POSITIONS = ["QB", "RB", "WR", "TE", "K", "DST"]

# --- Scoring -------------------------------------------------------------
# NOT modeled (each needs TD-level or play-level detail no projection
# source breaks out): 40+ yard TD bonuses, return TDs, offensive fumble
# return TDs, the extra -2 pick-six penalty, missed FG 0-19.
SCORING = {
    # passing
    "pass_yd": 0.04,       # 25 passing yards per point
    "pass_td": 6.0,        # league override (Yahoo default is 4)
    "int": -1.0,           # league value (2025 model wrongly used -2)
    "pass_40p": 1.0,       # 40+ yard completions
    # rushing
    "rush_yd": 0.1,        # 10 rushing yards per point
    "rush_td": 6.0,
    "rush_fd": 0.5,        # rushing 1st downs — now a REAL projected stat
    "rush_1d": 0.5,        # alias, kept so an un-replaced Cell 4 still runs
    "rush_40p": 2.0,       # 40+ yard runs
    # receiving
    "rec": 0.5,            # HALF-PPR (2025 model wrongly used full PPR)
    "rec_yd": 0.1,
    "rec_td": 6.0,
    "rec_40p": 1.0,        # 40+ yard receptions
    # misc
    "fumble": -2.0,
    "two_pt": 2.0,         # applies to pass/rush/rec conversions alike
    # kicker — exact distance tiers used when the source provides buckets,
    # otherwise fg_avg is the fallback approximation
    "fg_0_19": 3.0, "fg_20_29": 3.0, "fg_30_39": 3.0,
    "fg_40_49": 4.0, "fg_50p": 5.0,
    "fg_avg": 3.6,
    "xp": 1.0,
    # team defense
    "dst_sack": 1.0, "dst_int": 2.0, "dst_fr": 2.0,
    "dst_td": 6.0, "dst_safety": 2.0,
}

# Fallback only. Sleeper projects rush_fd directly, so these rates are used
# just in case that field is ever missing for a player.
RUSH_1D_RATES = {"QB": 0.08, "RB": 0.05, "WR": 0.05, "TE": 0.05, "K": 0.0, "DST": 0.0}

# --- Roster (drives replacement levels) ----------------------------------
ROSTER = {
    "teams": 12,
    "slots": {"QB": 1, "RB": 2, "WR": 3, "TE": 1, "K": 1, "DST": 1},
    "flex": 1,                                    # the W/R/T spot
    "flex_split": {"RB": 0.75, "WR": 0.25, "TE": 0.0},
}
# e.g. {"QB": 15} to model streaming; your league has no waiver priority
# and unlimited adds, so effective QB/TE/DST replacement runs a bit higher.
REPLACEMENT_OVERRIDE = {}


In [2]:
# ============================================================
# CELL 2 (REPLACEMENT) — SLEEPER PROJECTIONS API
#
# Why the rewrite: FantasyPros moved projections + ADP to client-side
# rendering in 2026. Their HTML now serves only 10 rows per position and
# the ADP table is gone entirely. Sleeper's public projections endpoint
# returns full-season stat lines AND half-PPR ADP as JSON — no auth, no
# HTML parsing, and no cross-source name matching (stats and ADP arrive
# in the same record).
#
# Bonus: Sleeper projects rush_fd (rushing first downs) directly, so the
# RUSH_1D_RATES estimate is no longer needed. It also gives rec_40p for
# your league's 40+ yard reception bonus, and gp (games played).
# ============================================================
SLEEPER_SEASON = "2026"
SLEEPER_URL = f"https://api.sleeper.app/projections/nfl/{SLEEPER_SEASON}"

# canonical column -> Sleeper stat key. Missing keys resolve to 0.0 and are
# listed in the discovery report, so a renamed field is visible, never silent.
STAT_MAP = {
    "PASS_YD": "pass_yd", "PASS_TD": "pass_td", "INT": "pass_int",
    "PASS_40P": "pass_40p", "PASS_2PT": "pass_2pt",
    "RUSH_ATT": "rush_att", "RUSH_YD": "rush_yd", "RUSH_TD": "rush_td",
    "RUSH_FD": "rush_fd", "RUSH_40P": "rush_40p", "RUSH_2PT": "rush_2pt",
    "REC": "rec", "REC_YD": "rec_yd", "REC_TD": "rec_td",
    "REC_FD": "rec_fd", "REC_40P": "rec_40p", "REC_2PT": "rec_2pt",
    "FL": "fum_lost", "GP": "gp",
    # kicker
    "FG": "fgm", "XPT": "xpm",
    "FG_0_19": "fgm_0_19", "FG_20_29": "fgm_20_29", "FG_30_39": "fgm_30_39",
    "FG_40_49": "fgm_40_49", "FG_50P": "fgm_50p",
    # team defense
    "DST_SACK": "sack", "DST_INT": "int", "DST_FR": "fum_rec",
    "DST_TD": "def_td", "DST_SAFETY": "safe", "DST_PA": "pts_allow",
    # source ADP + Sleeper's own scoring (reference only)
    "ADP_HALF": "adp_half_ppr", "SLEEPER_PTS_HALF": "pts_half_ppr",
}

ALL_STAT_COLS = list(STAT_MAP.keys())
SLEEPER_POS_MAP = {"DEF": "DST"}   # Sleeper calls team defense DEF
ADP_UNDRAFTED = 900.0              # Sleeper uses 999.0 as "no ADP" sentinel

def fetch_sleeper():
    """One call returns every projected player. We filter by position
    client-side rather than trusting the position[] query param."""
    params = [("season_type", "regular"), ("order_by", "adp_half_ppr")]
    for p in ("QB", "RB", "WR", "TE", "K", "DEF"):
        params.append(("position[]", p))
    r = requests.get(SLEEPER_URL, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    if not isinstance(data, list) or not data:
        raise RuntimeError(f"Sleeper returned unexpected payload: {type(data)}")
    return data

def sleeper_to_frame(records):
    """Flatten Sleeper records into the canonical schema the rest of this
    notebook already uses (Player / Team / POS / stat columns)."""
    rows, seen_keys = [], {}
    for rec in records:
        p = rec.get("player") or {}
        stats = rec.get("stats") or {}
        pos = p.get("position") or (p.get("fantasy_positions") or [None])[0]
        if not pos:
            continue
        pos = SLEEPER_POS_MAP.get(pos, pos)
        if pos not in POSITIONS:
            continue

        name = (p.get("full_name")
                or " ".join(filter(None, [p.get("first_name"), p.get("last_name")]))
                or p.get("last_name") or "").strip()
        if not name:
            continue

        row = {"Player": name,
               "Team": (p.get("team") or "FA"),
               "POS": pos,
               "SLEEPER_ID": rec.get("player_id") or p.get("player_id")}
        for col, key in STAT_MAP.items():
            row[col] = float(stats.get(key) or 0.0)
        rows.append(row)
        seen_keys.setdefault(pos, set()).update(stats.keys())

    df = pd.DataFrame(rows)
    # ADP: 999.0 means undrafted -> treat as missing
    df.loc[df["ADP_HALF"] >= ADP_UNDRAFTED, "ADP_HALF"] = float("nan")
    df.loc[df["ADP_HALF"] == 0.0, "ADP_HALF"] = float("nan")
    return df, seen_keys

def discovery_report(df, seen_keys):
    """Prints which mapped fields actually came back, and what else is
    available. Read this once after a season rollover or API change."""
    print("\n--- DISCOVERY REPORT ---")
    mapped = set(STAT_MAP.values())
    for pos in POSITIONS:
        keys = seen_keys.get(pos, set())
        if not keys:
            print(f"{pos}: NO RECORDS")
            continue
        missing = sorted(k for k in mapped if k not in keys)
        print(f"{pos}: {len(df[df['POS'] == pos])} players, {len(keys)} stat keys")
        if missing:
            print(f"   mapped-but-absent: {missing}")
    all_keys = sorted(set().union(*seen_keys.values())) if seen_keys else []
    unused = [k for k in all_keys if k not in mapped]
    print(f"\nAvailable but unused ({len(unused)}): {unused}")

In [3]:
# ============================================================
# CELL 3 (REPLACEMENT) — FETCH + SNAPSHOT
# Optionally overlays FantasyPros ECR/tier from the cheatsheets
# page, which still works (it's the only FP endpoint that does).
# ============================================================
USE_FANTASYPROS_ECR = True   # set False to run purely on Sleeper

records = fetch_sleeper()
print(f"Sleeper: {len(records)} raw records")
proj, seen_keys = sleeper_to_frame(records)
print(f"usable players: {len(proj)}")
discovery_report(proj, seen_keys)

# --- ADP straight from the same payload (no name matching needed) ---
adp = (proj.loc[proj["ADP_HALF"].notna(), ["Player", "Team", "POS", "ADP_HALF"]]
       .sort_values("ADP_HALF").reset_index(drop=True))
adp["ADP_RANK"] = adp.index + 1
adp = adp.rename(columns={"ADP_HALF": "AVG"})
adp["Bye"] = None
print(f"\nplayers with half-PPR ADP: {len(adp)}")

# --- optional FantasyPros ECR overlay (cheatsheets ecrData still works) ---
if USE_FANTASYPROS_ECR:
    def extract_js_json(html, varname):
        i = html.find(varname)
        if i == -1:
            raise ValueError(f"{varname} not found")
        s = html.find("=", i) + 1
        while html[s] in " \t\r\n":
            s += 1
        oc = html[s]; cc = "}" if oc == "{" else "]"
        depth = 0; j = s; in_str = False; esc = False
        while j < len(html):
            c = html[j]
            if in_str:
                if esc: esc = False
                elif c == "\\": esc = True
                elif c == '"': in_str = False
            else:
                if c == '"': in_str = True
                elif c == oc: depth += 1
                elif c == cc:
                    depth -= 1
                    if depth == 0:
                        return json.loads(html[s:j + 1])
            j += 1
        raise ValueError(f"unterminated {varname}")

    fp_html = requests.get(
        "https://www.fantasypros.com/nfl/rankings/half-point-ppr-cheatsheets.php",
        headers=HEADERS, timeout=30).text
    ecr = extract_js_json(fp_html, "ecrData")
    fp = pd.DataFrame([{
        "Player": p["player_name"],
        "POS": SLEEPER_POS_MAP.get(p["player_position_id"], p["player_position_id"]),
        "ECR": p.get("rank_ecr"),
        "FP_TIER": p.get("tier"),
        "Bye": p.get("player_bye_week"),
    } for p in ecr["players"]])
    print(f"FantasyPros ECR: {len(fp)} players (as of {ecr.get('last_updated')})")
else:
    fp = None

stamp = date.today().isoformat()
proj.to_csv(f"projections_{stamp}.csv", index=False)
adp.to_csv(f"adp_{stamp}.csv", index=False)
print(f"snapshots saved: projections_{stamp}.csv, adp_{stamp}.csv")

# Only players with real rushing volume tell us whether the field is populated.
_r = proj[(proj["POS"].isin(["QB", "RB"])) & (proj["RUSH_ATT"] >= 20)]
share = (_r["RUSH_FD"] > 0).mean()
RUSH_FD_OK = bool(share > 0.8)
print(f"rush_fd populated for {share:.0%} of {len(_r)} QB/RB with 20+ carries "
      f"-> RUSH_FD_OK={RUSH_FD_OK}")

Sleeper: 3301 raw records
usable players: 3226

--- DISCOVERY REPORT ---
QB: 355 players, 31 stat keys
   mapped-but-absent: ['def_td', 'fgm', 'fgm_0_19', 'fgm_20_29', 'fgm_30_39', 'fgm_40_49', 'fgm_50p', 'fum_rec', 'int', 'pass_40p', 'pts_allow', 'rec', 'rec_2pt', 'rec_40p', 'rec_fd', 'rec_td', 'rec_yd', 'rush_40p', 'sack', 'safe', 'xpm']
RB: 677 players, 35 stat keys
   mapped-but-absent: ['def_td', 'fgm', 'fgm_0_19', 'fgm_20_29', 'fgm_30_39', 'fgm_40_49', 'fgm_50p', 'fum_rec', 'int', 'pass_2pt', 'pass_40p', 'pass_int', 'pass_td', 'pass_yd', 'pts_allow', 'rec_2pt', 'rush_40p', 'sack', 'safe', 'xpm']
WR: 1365 players, 40 stat keys
   mapped-but-absent: ['def_td', 'fgm', 'fgm_0_19', 'fgm_20_29', 'fgm_30_39', 'fgm_40_49', 'fgm_50p', 'fum_rec', 'int', 'pass_2pt', 'pass_40p', 'pass_int', 'pass_td', 'pass_yd', 'pts_allow', 'rush_2pt', 'rush_40p', 'sack', 'safe', 'xpm']
TE: 644 players, 34 stat keys
   mapped-but-absent: ['def_td', 'fgm', 'fgm_0_19', 'fgm_20_29', 'fgm_30_39', 'fgm_40_49', '

In [4]:
def probe(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    soup = BeautifulSoup(r.text, "html.parser")
    print("Requested :", url)
    print("Final URL :", r.url)                      # reveals redirects
    print("Status/size:", r.status_code, "/", len(r.text), "bytes")
    print("Title     :", soup.title.get_text(strip=True) if soup.title else "none")
    print("Tables    :", [(t.get("id"), " ".join(t.get("class", [])), len(t.find_all("tr")))
                          for t in soup.find_all("table")])
    for key in ("ecrData", "adpData", "__NEXT_DATA__", "__NUXT__"):
        if key in r.text:
            print("Embedded JSON:", key, "FOUND")
    print("-" * 60)

probe("https://www.fantasypros.com/nfl/projections/rb.php?week=draft&scoring=HALF")
probe("https://www.fantasypros.com/nfl/projections/rb.php?week=draft")
probe("https://www.fantasypros.com/nfl/adp/half-point-ppr-overall.php")
probe("https://www.fantasypros.com/nfl/adp/ppr-overall.php")

Requested : https://www.fantasypros.com/nfl/projections/rb.php?week=draft&scoring=HALF
Final URL : https://www.fantasypros.com/nfl/projections/rb.php?week=draft
Status/size: 200 / 303745 bytes
Title     : 2026 Half-PPR RB Projections - Consensus Fantasy Football Stats for Running Backs | FantasyPros
Tables    : [('data', 'table table--sticky-columns table-bordered table-striped table-hover', 12)]
------------------------------------------------------------
Requested : https://www.fantasypros.com/nfl/projections/rb.php?week=draft
Final URL : https://www.fantasypros.com/nfl/projections/rb.php?week=draft
Status/size: 200 / 303745 bytes
Title     : 2026 Half-PPR RB Projections - Consensus Fantasy Football Stats for Running Backs | FantasyPros
Tables    : [('data', 'table table--sticky-columns table-bordered table-striped table-hover', 12)]
------------------------------------------------------------
Requested : https://www.fantasypros.com/nfl/adp/half-point-ppr-overall.php
Final URL : http

In [5]:
# ============================================================
# CELL 3b — PLAYER METADATA (age, experience, injury, depth chart)
# Sleeper's projection records carry a nested `player` object. We only
# used position/team/name from it; this grabs the rest.
# ============================================================
print("player object fields:", sorted((records[0].get("player") or {}).keys()))

META_FIELDS = ["age", "years_exp", "injury_status", "status",
               "depth_chart_order", "number", "height", "weight"]

meta = []
for rec in records:
    p = rec.get("player") or {}
    row = {"SLEEPER_ID": rec.get("player_id") or p.get("player_id")}
    for f in META_FIELDS:
        row[f.upper()] = p.get(f)
    meta.append(row)

meta = pd.DataFrame(meta).drop_duplicates("SLEEPER_ID")
proj = proj.merge(meta, on="SLEEPER_ID", how="left")

for c in ("AGE", "YEARS_EXP", "DEPTH_CHART_ORDER"):
    if c in proj.columns:
        proj[c] = pd.to_numeric(proj[c], errors="coerce")

print(f"\nage populated for {proj['AGE'].notna().mean():.0%} of players")
print(proj.groupby("POS")["AGE"].median().round(1))

player object fields: ['fantasy_positions', 'first_name', 'injury_body_part', 'injury_notes', 'injury_start_date', 'injury_status', 'last_name', 'metadata', 'news_updated', 'position', 'team', 'team_abbr', 'team_changed_at', 'years_exp']

age populated for 0% of players
POS
DST   NaN
K     NaN
QB    NaN
RB    NaN
TE    NaN
WR    NaN
Name: AGE, dtype: float64


In [6]:
# ============================================================
# CELL 3c — PLAYER METADATA (Sleeper) + AGE (nflverse)
# Sleeper's player object has no age field, so age comes from nflverse.
# Self-contained: defines its own loader.
# ============================================================
NFLVERSE = "https://github.com/nflverse/nflverse-data/releases/download"

def load_first(candidates, label):
    """Try candidate URLs in order; return the first that loads."""
    for url in candidates:
        try:
            df = (pd.read_parquet(url) if url.endswith(".parquet")
                  else pd.read_csv(url, low_memory=False))
            print(f"OK  {label}: {url.split('/')[-1]}  ({len(df):,} rows)")
            return df
        except Exception as e:
            print(f"    no: {url.split('/')[-1]}  ({type(e).__name__})")
    raise RuntimeError(f"could not load {label}")

# ---- 1. metadata Sleeper actually provides ----
META_FIELDS = ["years_exp", "injury_status", "injury_body_part",
               "injury_notes", "team_changed_at"]

meta = []
for rec in records:
    p = rec.get("player") or {}
    row = {"SLEEPER_ID": rec.get("player_id") or p.get("player_id")}
    for f in META_FIELDS:
        row[f.upper()] = p.get(f)
    meta.append(row)

meta = pd.DataFrame(meta).drop_duplicates("SLEEPER_ID")
proj = proj.drop(columns=[f.upper() for f in META_FIELDS], errors="ignore")
proj = proj.merge(meta, on="SLEEPER_ID", how="left")
proj["YEARS_EXP"] = pd.to_numeric(proj["YEARS_EXP"], errors="coerce")
print(f"years_exp populated for {proj['YEARS_EXP'].notna().mean():.0%} of players")

# ---- 2. age from nflverse ----
def _norm(n):
    n = str(n).lower().replace(".", "").replace("'", "")
    n = re.sub(r"\s+(jr|sr|ii|iii|iv|v)$", "", n)
    return re.sub(r"\s+", " ", n).strip()

ros = load_first([
    f"{NFLVERSE}/players/players.parquet",          # all players, year-independent
    f"{NFLVERSE}/rosters/roster_2025.parquet",
    f"{NFLVERSE}/rosters/rosters_2025.parquet",
    f"{NFLVERSE}/rosters/roster_2024.parquet",
], "player birth dates")

print("name/birth columns:",
      [c for c in ros.columns if "name" in c.lower() or "birth" in c.lower()])

namecol = next((c for c in ("display_name", "player_name", "full_name", "football_name")
                if c in ros.columns), None)
birthcol = next((c for c in ("birth_date", "birthdate") if c in ros.columns), None)
if namecol is None or birthcol is None:
    raise RuntimeError(f"unexpected roster schema — columns: {list(ros.columns)[:40]}")

ros = ros[[namecol, birthcol]].dropna()
ros[birthcol] = pd.to_datetime(ros[birthcol], errors="coerce")
ros["AGE"] = (pd.Timestamp("2026-09-01") - ros[birthcol]).dt.days / 365.25
ros = ros[(ros["AGE"] > 18) & (ros["AGE"] < 50)]          # drop bad birth dates
ros["_k"] = ros[namecol].map(_norm)
ros = ros.drop_duplicates("_k")[["_k", "AGE"]]

proj["_k"] = proj["Player"].map(_norm)
proj = proj.drop(columns=["AGE"], errors="ignore").merge(ros, on="_k", how="left")

drafted = proj[proj["ADP_HALF"].notna()]
print(f"\nage matched for {proj['AGE'].notna().mean():.0%} of all players, "
      f"{drafted['AGE'].notna().mean():.0%} of players with ADP")
print(drafted.groupby("POS")["AGE"].median().round(1))

years_exp populated for 99% of players
OK  player birth dates: players.parquet  (25,048 rows)
name/birth columns: ['display_name', 'common_first_name', 'first_name', 'last_name', 'short_name', 'football_name', 'birth_date', 'college_name']

age matched for 72% of all players, 89% of players with ADP
POS
DST     NaN
K      28.8
QB     28.5
RB     26.7
TE     27.3
WR     26.3
Name: AGE, dtype: float64


In [7]:
# ============================================================
# CELL 4 — CUSTOM SCORING ON REAL PROJECTED STATS  (complete)
#
# Skill positions are scored from raw stats with your exact league
# settings. K and DST fall back to Sleeper's own scoring because the
# payload omits the components we'd need (no fgm total, no pts_allow) —
# and VOR spread at those two positions is small enough that the
# approximation costs nothing on draft day.
#
# Categories your league pays for but Sleeper does NOT project:
#   rush_40p (40+ yard runs, 2 pts) and pass_40p (40+ completions, 1 pt)
#   are absent for every position, so those terms contribute 0. They're
#   left in the formula so they start counting automatically if Sleeper
#   ever adds them. rec_40p IS projected and is credited.
# Also unmodeled (need TD-level detail no source breaks out): 40+ yard
#   TD bonuses, return TDs, offensive fumble return TDs, pick-six extra
#   penalty, missed FG 0-19.
# ============================================================
def score_row(row):
    def g(col):
        """Safe stat read: missing or NaN -> 0.0. Without the NaN guard a
        single blank field would silently turn PROJ_PTS into NaN and drop
        the player out of the VOR ordering entirely."""
        v = row.get(col, 0.0)
        try:
            v = float(v)
        except (TypeError, ValueError):
            return 0.0
        return 0.0 if pd.isna(v) else v

    # --- kickers and team defenses: use Sleeper's own points ---
    if row["POS"] in ("K", "DST"):
        return g("SLEEPER_PTS_HALF")

    # --- rushing first downs: real projection, estimate only as fallback ---
    rush_fd = (g("RUSH_FD") if RUSH_FD_OK
               else g("RUSH_YD") * RUSH_1D_RATES.get(row["POS"], 0.0))

    return (
        # passing
        g("PASS_YD") * SCORING["pass_yd"]
        + g("PASS_TD") * SCORING["pass_td"]
        + g("INT") * SCORING["int"]
        + g("PASS_40P") * SCORING["pass_40p"]        # not projected -> 0
        # rushing
        + g("RUSH_YD") * SCORING["rush_yd"]
        + g("RUSH_TD") * SCORING["rush_td"]
        + rush_fd * SCORING["rush_fd"]
        + g("RUSH_40P") * SCORING["rush_40p"]        # not projected -> 0
        # receiving
        + g("REC") * SCORING["rec"]
        + g("REC_YD") * SCORING["rec_yd"]
        + g("REC_TD") * SCORING["rec_td"]
        + g("REC_40P") * SCORING["rec_40p"]
        # misc
        + g("FL") * SCORING["fumble"]
        + (g("PASS_2PT") + g("RUSH_2PT") + g("REC_2PT")) * SCORING["two_pt"]
    )


proj["PROJ_PTS"] = proj.apply(score_row, axis=1).round(1)

# --- sanity check: top of each position should look plausible ---
for pos in POSITIONS:
    top = (proj[proj["POS"] == pos]
           .nlargest(3, "PROJ_PTS")[["Player", "PROJ_PTS"]]
           .to_string(index=False, header=False)
           .replace("\n", " | "))
    print(f"{pos:>4}: {top}")

  QB:    Josh Allen 442.2 | Lamar Jackson 413.4 |    Drake Maye 399.6
  RB:    Jahmyr Gibbs 368.8 |  Bijan Robinson 367.9 | Jonathan Taylor 327.2
  WR:         Puka Nacua 272.4 |      Ja'Marr Chase 268.3 | Jaxon Smith-Njigba 246.6
  TE:     Brock Bowers 213.6 |     Trey McBride 196.5 | Colston Loveland 182.0
   K:   Brandon Aubrey 116.0 | Ka'imi Fairbairn 113.0 |       Cam Little 112.0
 DST: Los Angeles Rams 106.0 |   Houston Texans 104.0 | Seattle Seahawks 103.0


In [8]:
# ============================================================
# CELL 6 — REPLACEMENT LEVELS + VOR
# Replacement rank is derived from roster structure and recomputed
# from live data every run:
#   last starter = teams x slots + this position's share of flexes
#   replacement  = the next player down
# For this league: QB13, RB31, WR43, TE13, K13, DST13.
# ============================================================
def replacement_ranks():
    out = {}
    for pos in POSITIONS:
        starters = ROSTER["teams"] * ROSTER["slots"].get(pos, 0)
        flex = (round(ROSTER["teams"] * ROSTER["flex"] * ROSTER["flex_split"].get(pos, 0.0))
                if pos in ("RB", "WR", "TE") else 0)
        out[pos] = REPLACEMENT_OVERRIDE.get(pos, starters + flex + 1)
    return out

repl_rank = replacement_ranks()
repl_pts = {}
for pos in POSITIONS:
    pool = proj.loc[proj["POS"] == pos].sort_values("PROJ_PTS", ascending=False)
    if len(pool) == 0:
        repl_pts[pos] = 0.0
        continue
    idx = min(repl_rank[pos] - 1, len(pool) - 1)      # clamp if pool is shallow
    repl_pts[pos] = float(pool["PROJ_PTS"].iloc[idx])

print("replacement levels:",
      {p: f"{p}{repl_rank[p]} = {repl_pts[p]:.1f} pts" for p in POSITIONS})

proj["VOR"] = proj.apply(lambda r: r["PROJ_PTS"] - repl_pts[r["POS"]], axis=1).round(1)
proj = proj.sort_values("VOR", ascending=False).reset_index(drop=True)
proj["VOR_RANK"] = proj.index + 1
proj["POS_RANK"] = proj.groupby("POS")["PROJ_PTS"].rank(ascending=False,
                                                        method="first").astype(int)

replacement levels: {'QB': 'QB13 = 364.2 pts', 'RB': 'RB34 = 177.4 pts', 'WR': 'WR40 = 146.7 pts', 'TE': 'TE13 = 135.8 pts', 'K': 'K13 = 103.0 pts', 'DST': 'DST13 = 86.0 pts'}


In [9]:
# ============================================================
# CELL 7 — TIERS (gap-based, per position)
# A new tier starts wherever the drop to the next player is unusually
# large (> 1.25x the average gap near the top of the position, with a
# 4-point floor so flat tails don't fragment). Draft by tier: "how many
# are left in this tier vs. the next tier at another position?"
# ============================================================
def assign_tiers(points_desc):
    n = len(points_desc)
    if n == 0:
        return []
    gaps = [points_desc[i - 1] - points_desc[i] for i in range(1, min(n, 40))]
    mean_gap = sum(gaps) / len(gaps) if gaps else 0.0
    threshold = max(mean_gap * 1.25, 4.0)
    tiers, tier = [1], 1
    for i in range(1, n):
        if points_desc[i - 1] - points_desc[i] > threshold:
            tier += 1
        tiers.append(tier)
    return tiers

proj["TIER"] = 0
for pos in POSITIONS:
    pool = proj.loc[proj["POS"] == pos].sort_values("PROJ_PTS", ascending=False)
    proj.loc[pool.index, "TIER"] = assign_tiers(pool["PROJ_PTS"].tolist())

In [10]:
# ============================================================
# CELL 8 — MERGE ADP + FANTASYPROS ECR
# Sleeper omits name suffixes (Jr./Sr./II/III); FantasyPros includes
# them, so both sides are matched on a normalized key. Two passes:
# name+position first, then name-only for anyone left (position
# disagreements between sources are common for hybrid players).
# ============================================================
def _norm(n):
    n = str(n).lower().replace(".", "").replace("'", "").replace("-", " ")
    n = re.sub(r"\s+(jr|sr|ii|iii|iv|v)$", "", n)
    return re.sub(r"\s+", " ", n).strip()

# manual overrides for genuine name differences (nicknames, etc.)
# key = normalized Sleeper name, value = normalized FantasyPros name
NAME_ALIASES = {
    # "hollywood brown": "marquise brown",
}

# --- ADP (same source as projections, so plain names are safe) ---
board = proj.merge(adp[["Player", "POS", "ADP_RANK", "AVG"]],
                   on=["Player", "POS"], how="left")

# --- FantasyPros ECR overlay ---
if fp is not None:
    board["_k"] = board["Player"].map(_norm).replace(NAME_ALIASES)
    fp = fp.copy()
    fp["_k"] = fp["Player"].map(_norm)

    fp_kp = fp.dropna(subset=["_k"]).drop_duplicates(["_k", "POS"]).set_index(["_k", "POS"])
    fp_k = fp.dropna(subset=["_k"]).drop_duplicates("_k").set_index("_k")
    FP_COLS = ["ECR", "FP_TIER", "Bye"]

    # pass 1: normalized name + position
    idx = pd.MultiIndex.from_arrays([board["_k"], board["POS"]])
    for c in FP_COLS:
        board[c] = fp_kp[c].reindex(idx).values

    # pass 2: normalized name only, for whatever pass 1 missed
    miss = board["ECR"].isna()
    for c in FP_COLS:
        board.loc[miss, c] = fp_k[c].reindex(board.loc[miss, "_k"]).values

    matched = board["ECR"].notna()
    print(f"ECR matched: {int(matched.sum())} of {len(board)} players")

    still = board[board["ADP_RANK"].notna() & (board["ADP_RANK"] <= 170)
                  & board["ECR"].isna()]
    if len(still):
        print(f"\n[!] {len(still)} top-170 players STILL missing ECR "
              f"(add to NAME_ALIASES if these are real players):")
        print(still[["Player", "POS", "Team", "ADP_RANK"]]
              .sort_values("ADP_RANK").to_string(index=False))
    else:
        print("all top-170 ADP players matched an ECR ✔")
else:
    board["ECR"] = float("nan")
    board["FP_TIER"] = float("nan")
    board["Bye"] = None

# --- draft-day value columns ---
# VALUE  = vs Sleeper ADP (sharp drafters)
# ROOM+  = vs FantasyPros ECR (closest proxy for your Yahoo room's board)
board["VALUE"] = (board["ADP_RANK"] - board["VOR_RANK"]).round(0)
board["ROOM+"] = (board["ECR"] - board["VOR_RANK"]).round(0)

ECR matched: 853 of 3226 players

[!] 1 top-170 players STILL missing ECR (add to NAME_ALIASES if these are real players):
        Player POS Team  ADP_RANK
Jayden Higgins  WR  HOU     140.0


In [19]:
# ============================================================
# CELL 8b — EMPIRICAL AVAILABILITY (games played by position + age)
#
# Sleeper projects a flat 18 games for everyone, so availability has to
# come from history. Two things make this non-trivial:
#
#   1. POPULATION. nflverse weekly data includes every fringe player who
#      ever recorded a stat, so a raw positional mean measures "games a
#      random NFL body plays" (~10 of 17), not "games a draftable starter
#      plays" (~13-16). We condition on a draftable cohort instead.
#
#   2. SELECTION. The cohort must be defined EX-ANTE. Filtering on the
#      same season's production would drop the Week-1 ACL tear — exactly
#      the outcome we're trying to measure — and bias availability up.
#      So the cohort for season Y is the top-N at each position in Y-1,
#      and players who then missed all of Y are kept as genuine zeros.
#
# Note: `games` conflates injury with losing your job. That's intended —
# a benched starter is equally useless to you, and VOR_ADJ backfills
# either case from waivers. Read the output as "expected games as a
# useful starter," not as an injury forecast.
#
# Slow cell: downloads ~5 seasons of weekly data. Run once.
# ============================================================
SEASONS = [2021, 2022, 2023, 2024, 2025]        # 17-game era only

wk = []
for y in SEASONS:
    try:
        d = load_first([
            f"{NFLVERSE}/player_stats/player_stats_{y}.parquet",
            f"{NFLVERSE}/stats_player/stats_player_week_{y}.parquet",
        ], f"weekly {y}")
        d["season"] = y
        wk.append(d)
    except Exception as e:
        print(f"skip {y}: {e}")
wk = pd.concat(wk, ignore_index=True)

pcol = "position" if "position" in wk.columns else "position_group"
ncol = "player_display_name" if "player_display_name" in wk.columns else "player_name"
ptscol = "fantasy_points_ppr" if "fantasy_points_ppr" in wk.columns else "fantasy_points"
SKILL = ["QB", "RB", "WR", "TE"]

# games actually played, per player-season (also used by Cell 8c)
gp = (wk[wk[pcol].isin(SKILL)]
      .groupby([ncol, pcol, "season"])["week"].nunique().reset_index(name="games"))

# season fantasy totals -> positional rank, for defining the cohort
season_pts = (wk[wk[pcol].isin(SKILL)]
              .groupby([ncol, pcol, "season"], as_index=False)[ptscol].sum())
season_pts["pos_rank"] = (season_pts.groupby([pcol, "season"])[ptscol]
                          .rank(ascending=False, method="first"))


def build_cohort(mult):
    """Cohort for season Y = top-N at position in Y-1, where N is this
    league's replacement rank scaled by `mult`. Returns one row per
    player-season with games played (0 if they never appeared)."""
    n = {p: max(int(repl_rank[p] * mult), 12) for p in SKILL}
    c = season_pts[season_pts["pos_rank"] <= season_pts[pcol].map(n)].copy()
    c["season"] += 1                                  # carry forward one year
    c = c[c["season"].isin(SEASONS)][[ncol, pcol, "season"]]
    c = c.merge(gp[[ncol, pcol, "season", "games"]],
                on=[ncol, pcol, "season"], how="left")
    c["games"] = c["games"].fillna(0.0)
    c["avail"] = c["games"] / 17.0
    return c, n


# ---- sensitivity: does the cohort threshold actually matter? ----
print("SENSITIVITY — availability rate by cohort width\n")
for m in (1.0, 1.5, 2.0, 3.0):
    c, n = build_cohort(m)
    rates = (c.groupby(pcol)["avail"].mean().round(3)).to_dict()
    print(f"  mult {m}: N={n}  ->  {rates}")
print("\n(if these barely move, the threshold doesn't matter; if RB slides as")
print(" the cohort widens, that's job-loss noise entering the sample)\n")

# ---- build the table at the chosen width ----
DRAFT_BUFFER = 1.5          # replacement rank x this = draftable pool
cohort, DRAFTABLE = build_cohort(DRAFT_BUFFER)
print(f"using DRAFT_BUFFER={DRAFT_BUFFER} -> {DRAFTABLE}")

# age at that season, from nflverse birth dates
pl = load_first([f"{NFLVERSE}/players/players.parquet"], "players")
pn = next(c for c in ("display_name", "player_name", "full_name") if c in pl.columns)
pl = pl[[pn, "birth_date"]].dropna().copy()
pl["birth_date"] = pd.to_datetime(pl["birth_date"], errors="coerce")
pl["_k"] = pl[pn].map(_norm)

cohort["_k"] = cohort[ncol].map(_norm)
cohort = cohort.merge(pl.drop_duplicates("_k")[["_k", "birth_date"]], on="_k", how="left")
cohort["age"] = ((pd.to_datetime(cohort["season"].astype(str) + "-09-01")
                  - cohort["birth_date"]).dt.days / 365.25)

BUCKETS = [(0, 26, "<26"), (26, 30, "26-30"), (30, 60, "30+")]
cohort["bucket"] = pd.cut(cohort["age"], [b[0] for b in BUCKETS] + [60],
                          labels=[b[2] for b in BUCKETS], right=False)

print("\nGAMES PLAYED BY DRAFTABLE STARTERS (of 17):")
print(cohort.groupby(pcol)["games"].describe()[["count", "mean", "50%", "min"]].round(1))

tbl = (cohort.dropna(subset=["bucket"])
       .groupby([pcol, "bucket"], observed=True)["avail"]
       .agg(["mean", "count"]).round(3))
print("\nAVAILABILITY BY POSITION AND AGE:")
print(tbl)

# buckets thinner than 25 observations fall back to the position mean
MIN_COUNT = 15          # coarser bands mean fewer, better-populated cells
AVAIL = {(p, b): r["mean"] for (p, b), r in tbl.iterrows() if r["count"] >= MIN_COUNT}
POS_MEAN = cohort.groupby(pcol)["avail"].mean().to_dict()
print(f"\nusable buckets: {len(AVAIL)} of {len(tbl)}")
print("position means:", {k: round(v, 3) for k, v in POS_MEAN.items()})
print("age matched for", f"{cohort['age'].notna().mean():.0%}", "of cohort")

OK  weekly 2021: player_stats_2021.parquet  (5,698 rows)
OK  weekly 2022: player_stats_2022.parquet  (5,631 rows)
OK  weekly 2023: player_stats_2023.parquet  (5,653 rows)
OK  weekly 2024: player_stats_2024.parquet  (5,597 rows)
    no: player_stats_2025.parquet  (HTTPError)
OK  weekly 2025: stats_player_week_2025.parquet  (19,422 rows)
SENSITIVITY — availability rate by cohort width

  mult 1.0: N={'QB': 13, 'RB': 34, 'WR': 40, 'TE': 13}  ->  {'QB': 0.876, 'RB': 0.782, 'TE': 0.852, 'WR': 0.835}
  mult 1.5: N={'QB': 19, 'RB': 51, 'WR': 60, 'TE': 19}  ->  {'QB': 0.838, 'RB': 0.782, 'TE': 0.836, 'WR': 0.79}
  mult 2.0: N={'QB': 26, 'RB': 68, 'WR': 80, 'TE': 26}  ->  {'QB': 0.768, 'RB': 0.707, 'TE': 0.799, 'WR': 0.765}
  mult 3.0: N={'QB': 39, 'RB': 102, 'WR': 120, 'TE': 39}  ->  {'QB': 0.664, 'RB': 0.57, 'TE': 0.754, 'WR': 0.676}

(if these barely move, the threshold doesn't matter; if RB slides as
 the cohort widens, that's job-loss noise entering the sample)

using DRAFT_BUFFER=1.5 -> {

In [12]:
# ============================================================
# CELL 8c — AVAILABILITY-ADJUSTED VOR  (display only)
#
# Missed games are backfilled from waivers at replacement level, which
# contributes zero VOR by definition. So expected value from the roster
# spot is G*(p - r), and since raw VOR is 18*(p - r):
#       VOR_ADJ = VOR * (expected_games / 18)
# Availability scales VOR, not points. VOR remains the primary ranking;
# VOR_ADJ is a second read on who is carrying risk.
# ============================================================
def _bucket(a):
    if pd.isna(a):
        return None
    for lo, hi, lab in BUCKETS:
        if lo <= a < hi:
            return lab
    return None

def _avail(pos, age):
    if pos in ("K", "DST"):
        return 1.0                      # streamed weekly; no availability drag
    b = _bucket(age)
    if b is not None and (pos, b) in AVAIL:
        return AVAIL[(pos, b)]
    return POS_MEAN.get(pos, 0.88)

board["EXP_AVAIL"] = [_avail(p, a) for p, a in zip(board["POS"], board["AGE"])]
board["EXP_GAMES"] = (board["EXP_AVAIL"] * 18).round(1)
board["VOR_ADJ"] = (board["VOR"] * board["EXP_AVAIL"]).round(1)

# strongest per-player signal: games actually played last season
last = gp[gp["season"] == 2025].drop_duplicates("_k").set_index("_k")["games"]
board["GP_2025"] = last.reindex(board["_k"]).values

# who moves most once availability is priced in
mv = board[board["ADP_RANK"].notna()].copy()
mv["ADJ_RANK"] = mv["VOR_ADJ"].rank(ascending=False, method="first")
mv["DROP"] = (mv["ADJ_RANK"] - mv["VOR_RANK"]).round(0)
print("BIGGEST AVAILABILITY DISCOUNTS (fall furthest when risk is priced in):")
print(mv.nlargest(15, "DROP")[
    ["Player", "POS", "AGE", "VOR", "VOR_ADJ", "EXP_GAMES", "GP_2025", "DROP"]
].to_string(index=False))

BIGGEST AVAILABILITY DISCOUNTS (fall furthest when risk is priced in):
             Player POS       AGE    VOR  VOR_ADJ  EXP_GAMES  GP_2025  DROP
  Noah Rauschenberg   K       NaN -103.0   -103.0       18.0      NaN 282.0
      James McCourt   K 28.810404 -103.0   -103.0       18.0      NaN 281.0
         Trey Wolff   K       NaN -103.0   -103.0       18.0      NaN 279.0
        Kai Forbath   K 38.997947 -103.0   -103.0       18.0      NaN 278.0
       Alex Kessman   K 28.662560 -103.0   -103.0       18.0      NaN 266.0
         Dan Bailey   K 38.598220 -103.0   -103.0       18.0      NaN 266.0
       Chris Naggar   K 28.728268 -103.0   -103.0       18.0      NaN 266.0
       Andrew Mevis   K       NaN -103.0   -103.0       18.0      NaN 263.0
    Spencer Shrader   K 27.288159 -103.0   -103.0       18.0      NaN 259.0
     Adam Vinatieri   K       NaN -103.0   -103.0       18.0      NaN 251.0
       Quinn Nordin   K 28.043806 -103.0   -103.0       18.0      NaN 250.0
          Cade Yo

In [20]:
# ============================================================
# CELL 9 — FINAL DRAFT SHEET
# VOR is the ranking. VOR_ADJ / EXP_GAMES are the availability read.
# ============================================================
SHEET_COLS = ["VOR_RANK", "Player", "POS", "POS_RANK", "Team", "AGE", "Bye", "TIER",
              "PROJ_PTS", "VOR", "VOR_ADJ", "EXP_GAMES",
              "ADP_RANK", "VALUE", "ECR", "ROOM+"]
sheet = board[[c for c in SHEET_COLS if c in board.columns]]

pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 200)
sheet.head(200)

,VOR_RANK,Player,POS,POS_RANK,Team,AGE,Bye,TIER,PROJ_PTS,VOR,VOR_ADJ,EXP_GAMES,ADP_RANK,VALUE,ECR,ROOM+
0,1,Jahmyr Gibbs,RB,1,DET,24.451745,6,1,368.8,191.4,116.2,10.9,1.0,0.0,1.0,0.0
1,2,Bijan Robinson,RB,2,ATL,24.585900,11,1,367.9,190.5,115.6,10.9,2.0,0.0,2.0,0.0
2,3,Jonathan Taylor,RB,3,IND,27.616701,13,2,327.2,149.8,89.3,10.7,5.0,2.0,7.0,4.0
3,4,Christian McCaffrey,RB,4,SF,30.234086,8,3,312.2,134.8,88.0,11.8,6.0,2.0,8.0,4.0
4,5,James Cook,RB,5,BUF,26.934976,7,3,311.9,134.5,80.2,10.7,9.0,4.0,11.0,6.0
5,6,Derrick Henry,RB,6,BAL,32.657084,13,3,310.4,133.0,69.7,9.4,18.0,12.0,22.0,16.0
6,7,Puka Nacua,WR,1,LAR,25.259411,11,1,272.4,125.7,69.5,10.0,4.0,-3.0,4.0,-3.0
7,8,Ja'Marr Chase,WR,2,CIN,26.502396,6,2,268.3,121.6,71.1,10.5,3.0,-5.0,3.0,-5.0
8,9,Ashton Jeanty,RB,7,LV,22.748802,13,4,297.1,119.7,68.8,10.4,11.0,2.0,12.0,3.0
9,10,De'Von Achane,RB,8,MIA,24.884326,6,4,293.8,116.4,70.7,10.9,14.0,4.0,18.0,8.0


In [14]:
# RUSH_1D estimator tool as a backup
import pandas as pd

NFLVERSE = "https://github.com/nflverse/nflverse-data/releases/download"

def load_first(candidates, label):
    """Try candidate URLs in order; return the first that loads."""
    for url in candidates:
        try:
            df = (pd.read_parquet(url) if url.endswith(".parquet")
                  else pd.read_csv(url, low_memory=False))
            print(f"OK  {label}: {url.split('/')[-1]}  ({len(df):,} rows)")
            return df
        except Exception as e:
            print(f"    no: {url.split('/')[-1]}  ({type(e).__name__})")
    raise RuntimeError(f"could not load {label}")

def season_stats(year):
    return load_first([
        f"{NFLVERSE}/player_stats/stats_player_season_{year}.parquet",
        f"{NFLVERSE}/player_stats/player_stats_{year}.parquet",
        f"{NFLVERSE}/player_stats/player_stats_{year}.csv.gz",
        f"{NFLVERSE}/stats_player/stats_player_week_{year}.parquet",
        f"{NFLVERSE}/stats_player/stats_player_season_{year}.parquet",
    ], f"season stats {year}")

# ---- fit rushing-first-down rates from actuals ----
frames = []
for y in (2022, 2023, 2024, 2025):
    try:
        frames.append(season_stats(y))
    except Exception as e:
        print(f"skip {y}: {e}")
s = pd.concat(frames, ignore_index=True)
pos_col = "position" if "position" in s.columns else "position_group"
key = "player_id" if "player_id" in s.columns else "player_display_name"

# weekly -> player-season totals
season = (s.groupby([key, pos_col, "season"], as_index=False)
            [["rushing_yards", "rushing_first_downs", "carries"]].sum())

# keep player-seasons with real rushing volume (matches who we actually draft)
sub = season[season["carries"] >= 20]

fit = sub.groupby(pos_col)[["rushing_yards", "rushing_first_downs", "carries"]].sum()
fit["fd_per_yard"] = fit["rushing_first_downs"] / fit["rushing_yards"]
fit["fd_per_att"] = fit["rushing_first_downs"] / fit["carries"]
fit["player_seasons"] = sub.groupby(pos_col).size()
print(fit[["fd_per_yard", "fd_per_att", "player_seasons"]].round(4))

    no: stats_player_season_2022.parquet  (HTTPError)
OK  season stats 2022: player_stats_2022.parquet  (5,631 rows)
    no: stats_player_season_2023.parquet  (HTTPError)
OK  season stats 2023: player_stats_2023.parquet  (5,653 rows)
    no: stats_player_season_2024.parquet  (HTTPError)
OK  season stats 2024: player_stats_2024.parquet  (5,597 rows)
    no: stats_player_season_2025.parquet  (HTTPError)
    no: player_stats_2025.parquet  (HTTPError)
    no: player_stats_2025.csv.gz  (HTTPError)
OK  season stats 2025: stats_player_week_2025.parquet  (19,422 rows)
          fd_per_yard  fd_per_att  player_seasons
position                                         
QB             0.0784      0.3647             155
RB             0.0519      0.2251             381
TE             0.0757      0.3488               3
WR             0.0429      0.2209               7


In [15]:
# ============================================================
# CELL 10 — explain(): why is this player ranked here?
# Decomposes a player's projected points into scoring categories and
# shows the context the scoring can't see (age, experience, injury,
# team change, workload). Reads from `board` if Cell 8 has run, so the
# ADP and FantasyPros ECR columns are available; otherwise from `proj`.
# ============================================================
def explain(name):
    src = board if "board" in globals() else proj
    m = src[src["Player"].str.contains(name, case=False, na=False)]
    if m.empty:
        print(f"no match for '{name}'")
        return
    r = m.iloc[0]

    def g(col):
        v = r.get(col, 0.0)
        try:
            v = float(v)
        except (TypeError, ValueError):
            return 0.0
        return 0.0 if pd.isna(v) else v

    def v(col):
        x = r.get(col) if col in r.index else None
        if x is None:
            return None
        if isinstance(x, float) and pd.isna(x):
            return None
        return x

    # ---------- headline ----------
    adp_txt = (f"ADP {v('ADP_RANK'):.0f}" if v("ADP_RANK") is not None
               else (f"ADP {g('ADP_HALF'):.1f}" if g("ADP_HALF") else "no ADP"))
    print(f"{r['Player']}  ({r['POS']}{r['POS_RANK']}, {r['Team']})")
    print(f"  {r['PROJ_PTS']:.1f} pts | VOR {r['VOR']:.1f} | overall #{r['VOR_RANK']} "
          f"| tier {r['TIER']} | {adp_txt}"
          + (f" | FP ECR {v('ECR'):.0f}" if v("ECR") is not None else ""))

    # ---------- context the scoring cannot see ----------
    bits = []
    if v("AGE") is not None:
        bits.append(f"age {float(v('AGE')):.1f}")
    if v("YEARS_EXP") is not None:
        e = int(float(v("YEARS_EXP")))
        bits.append("ROOKIE" if e == 0 else f"{e}y exp")
    if v("INJURY_BODY_PART"):
        bits.append(f"injury: {v('INJURY_STATUS') or 'listed'} ({v('INJURY_BODY_PART')})")
    elif v("INJURY_STATUS"):
        bits.append(f"injury: {v('INJURY_STATUS')}")
    if v("TEAM_CHANGED_AT"):
        bits.append(f"changed team {str(v('TEAM_CHANGED_AT'))[:10]}")
    bits.append(f"{g('GP'):.0f} games projected")
    if r["POS"] in ("RB", "WR", "TE"):
        bits.append(f"{g('RUSH_ATT') + g('REC'):.0f} touches")
    print("  " + " | ".join(bits))

    if v("INJURY_NOTES"):
        print(f"  note: {str(v('INJURY_NOTES'))[:160]}")

    # ---------- K / DST use Sleeper's own scoring ----------
    if r["POS"] in ("K", "DST"):
        print(f"  replacement {r['POS']}{repl_rank[r['POS']]}: "
              f"{repl_pts[r['POS']]:.1f} pts")
        print("  (scored with Sleeper's own points — the payload omits the "
              "components needed for your league's K/DST rules)")
        return

    # ---------- how much of this is YOUR league's scoring ----------
    sl = g("SLEEPER_PTS_HALF")
    if sl > 0:
        print(f"  league premium: {r['PROJ_PTS'] - sl:+.1f} pts vs standard "
              f"half-PPR ({sl:.1f}) — this is your edge over the draft room")

    # ---------- availability heuristics (Sleeper assumes everyone plays) ----------
    age = float(v("AGE")) if v("AGE") is not None else None
    if r["POS"] == "RB" and age is not None and age >= 28:
        print("  [!] RB age 28+ with no injury discount in the projection — "
              "haircut availability yourself")
    if r["POS"] in ("RB", "WR") and g("RUSH_ATT") + g("REC") >= 300:
        print("  [!] very high projected workload — upside, but concentrated risk")
    if v("TEAM_CHANGED_AT"):
        print("  [!] changed teams — projections lag on new situations, "
              "worth a manual look")

    # ---------- scoring breakdown (sums to PROJ_PTS) ----------
    parts = {
        "pass yds":   g("PASS_YD") * SCORING["pass_yd"],
        "pass TDs":   g("PASS_TD") * SCORING["pass_td"],
        "INTs":       g("INT") * SCORING["int"],
        "rush yds":   g("RUSH_YD") * SCORING["rush_yd"],
        "rush TDs":   g("RUSH_TD") * SCORING["rush_td"],
        "rush 1Ds":   (g("RUSH_FD") if RUSH_FD_OK
                       else g("RUSH_YD") * RUSH_1D_RATES.get(r["POS"], 0.0))
                      * SCORING["rush_fd"],
        "receptions": g("REC") * SCORING["rec"],
        "rec yds":    g("REC_YD") * SCORING["rec_yd"],
        "rec TDs":    g("REC_TD") * SCORING["rec_td"],
        "40+ recs":   g("REC_40P") * SCORING["rec_40p"],
        "2-pt conv":  (g("PASS_2PT") + g("RUSH_2PT") + g("REC_2PT")) * SCORING["two_pt"],
        "fumbles":    g("FL") * SCORING["fumble"],
    }
    print(f"  replacement {r['POS']}{repl_rank[r['POS']]}: {repl_pts[r['POS']]:.1f} pts")
    for k, val in sorted(parts.items(), key=lambda kv: -abs(kv[1])):
        if abs(val) > 0.05:
            print(f"     {k:<12} {val:7.1f}   ({val / r['PROJ_PTS']:5.1%})")


explain("Henry")

Derrick Henry  (RB6, BAL)
  310.4 pts | VOR 133.0 | overall #6 | tier 3 | ADP 18 | FP ECR 22
  age 32.7 | 10y exp | 18 games projected | 298 touches
  league premium: +72.0 pts vs standard half-PPR (238.4) — this is your edge over the draft room
  [!] RB age 28+ with no injury discount in the projection — haircut availability yourself
  replacement RB34: 177.4 pts
     rush yds       140.6   (45.3%)
     rush TDs        72.0   (23.2%)
     rush 1Ds        70.3   (22.6%)
     rec yds         13.3   ( 4.3%)
     receptions       8.5   ( 2.7%)
     rec TDs          6.0   ( 1.9%)
     fumbles         -4.0   (-1.3%)
     2-pt conv        2.0   ( 0.6%)
     40+ recs         1.7   ( 0.5%)


In [16]:
# biggest market discounts in this league's scoring (your sleepers)
sheet[sheet["ADP_RANK"].notna()].sort_values("VALUE", ascending=False).head(80)


,VOR_RANK,Player,POS,POS_RANK,Team,AGE,Bye,TIER,PROJ_PTS,VOR,VOR_ADJ,EXP_GAMES,ADP_RANK,VALUE,ECR,ROOM+
253,254,Andrei Iosivas,WR,84,CIN,26.880219,6,15,79.0,-67.7,-39.6,10.5,790.0,536.0,317.0,63.0
268,269,Evan Engram,TE,42,DEN,31.997262,10,9,58.3,-77.5,-45.6,10.6,788.0,519.0,244.0,-25.0
255,256,Michael Mayer,TE,35,LV,25.155373,13,9,67.2,-68.6,-34.5,9.1,774.0,518.0,326.0,70.0
285,286,Marvin Mims,WR,99,DEN,24.454483,10,16,60.9,-85.8,-47.4,10.0,789.0,503.0,283.0,-3.0
141,142,Atlanta Falcons,DST,21,ATL,NaN,11,2,83.0,-3.0,-3.0,18.0,637.0,495.0,290.0,148.0
135,136,Indianapolis Colts,DST,19,IND,NaN,13,2,84.0,-2.0,-2.0,18.0,631.0,495.0,313.0,177.0
269,270,KaVontae Turpin,WR,91,DAL,30.080767,14,16,66.9,-79.8,-46.3,10.4,765.0,495.0,370.0,100.0
286,287,DeMario Douglas,WR,100,NE,25.730322,11,16,60.6,-86.1,-47.6,10.0,777.0,490.0,327.0,40.0
170,171,Jason Sanders,K,24,NYJ,30.792608,13,1,91.0,-12.0,-12.0,18.0,659.0,488.0,574.0,403.0
304,305,Tyquan Thornton,WR,108,KC,26.067077,5,16,54.6,-92.1,-53.9,10.5,783.0,478.0,269.0,-36.0


In [17]:
# ============================================================
# FINAL CELL — PRINTABLE DRAFT SHEET (paper workflow)
# Tier breaks are drawn as separator lines so they survive printing.
# ROOM+ = ECR - VOR_RANK: how much later the room (drafting off
#   Yahoo/ECR-style ranks) will take him than your board says.
# ADJ   = VOR discounted for expected games played. If ADJ is well
#   below VOR, that value depends on him staying healthy.
# ============================================================
sheet = board[board["ADP_RANK"].notna() | board["ECR"].notna()].copy()

DEPTH = {"QB": 20, "RB": 45, "WR": 45, "TE": 18, "K": 14, "DST": 14}
HDR = ("rank  player                 tm age bye    proj    vor    adj   ecr room+")

def _fmt(r, with_pos=False):
    age = f"{r['AGE']:.0f}" if pd.notna(r["AGE"]) else "--"
    bye = f"{int(r['Bye'])}" if pd.notna(r["Bye"]) else "--"
    ecr = f"{r['ECR']:.0f}" if pd.notna(r["ECR"]) else "--"
    rp = f"{r['ROOM+']:+.0f}" if pd.notna(r["ROOM+"]) else "--"
    adj = f"{r['VOR_ADJ']:.1f}" if pd.notna(r.get("VOR_ADJ")) else "--"
    who = (f"{r['Player']:<22.22} {r['POS']}{int(r['POS_RANK']):<3}{r['Team']:<4}"
           if with_pos else f"{r['Player']:<22.22} {r['Team']:<4}")
    return (f"{int(r['VOR_RANK']):>4}  {who}{age:>3} {bye:>3}  "
            f"{r['PROJ_PTS']:>6.1f} {r['VOR']:>6.1f} {adj:>6}  {ecr:>4} {rp:>5}")

print(f"FREE STEAK DINNER 2026 — generated {stamp}")
print("VOR is the ranking. ADJ = VOR discounted for expected games played.")
print("ROOM+ positive = the room takes him later than he's worth; you can wait.")

print(f"\n{'#' * 22} OVERALL TOP 60 {'#' * 22}")
print("rank  player                 pos tm  age bye    proj    vor    adj   ecr room+")
for _, r in sheet.sort_values("VOR", ascending=False).head(60).iterrows():
    print(_fmt(r, with_pos=True))

for pos in ["RB", "WR", "QB", "TE", "K", "DST"]:
    p = sheet[sheet["POS"] == pos].sort_values("VOR", ascending=False).head(DEPTH[pos])
    if p.empty:
        continue
    print(f"\n{'=' * 26} {pos} {'=' * 26}")
    print(HDR)
    last_tier = None
    for _, r in p.iterrows():
        if last_tier is not None and r["TIER"] != last_tier:
            print("-" * 72)
        last_tier = r["TIER"]
        print(_fmt(r))

sheet.to_csv(f"printable_cheatsheet_{stamp}.csv", index=False)
print(f"\nsaved printable_cheatsheet_{stamp}.csv")

FREE STEAK DINNER 2026 — generated 2026-08-20
VOR is the ranking. ADJ = VOR discounted for expected games played.
ROOM+ positive = the room takes him later than he's worth; you can wait.

###################### OVERALL TOP 60 ######################
rank  player                 pos tm  age bye    proj    vor    adj   ecr room+
   1  Jahmyr Gibbs           RB1  DET  24   6   368.8  191.4  116.2     1    +0
   2  Bijan Robinson         RB2  ATL  25  11   367.9  190.5  115.6     2    +0
   3  Jonathan Taylor        RB3  IND  28  13   327.2  149.8   89.3     7    +4
   4  Christian McCaffrey    RB4  SF   30   8   312.2  134.8   88.0     8    +4
   5  James Cook             RB5  BUF  27   7   311.9  134.5   80.2    11    +6
   6  Derrick Henry          RB6  BAL  33  13   310.4  133.0   69.7    22   +16
   7  Puka Nacua             WR1  LAR  25  11   272.4  125.7   69.5     4    -3
   8  Ja'Marr Chase          WR2  CIN  27   6   268.3  121.6   71.1     3    -5
   9  Ashton Jeanty          RB7